# RMPC Lab 1 — Inverse Dynamics of a Robot Manipulator

**Course:** Robot Motion Planning and Control (ITMO University)  
**Objective:** Load a manipulator model (different from Puma560), specify dynamic parameters, plan a trajectory, solve the inverse dynamics problem using the Newton–Euler method, and analyse the M(q), C(q,q̇), G(q) matrices.

**Robot chosen:** Stanford Manipulator (6-DOF, RRP configuration — includes a prismatic joint)

## 0. Install dependencies

In [ ]:
!pip install roboticstoolbox-python swift-sim numpy matplotlib

In [ ]:
import roboticstoolbox as rtb
import numpy as np
import matplotlib.pyplot as plt
from spatialmath import SE3

np.set_printoptions(precision=4, suppress=True)

## 1. Load the manipulator model and display DH parameters

We use the **Stanford Arm** (also known as the Stanford Manipulator or Scheinman arm).  
It is a 6-DOF manipulator with an RRP (revolute-revolute-prismatic) shoulder/elbow structure,  
followed by a 3-DOF spherical wrist. This is explicitly **different from Puma560** as required.

In [ ]:
# Load the Stanford robot model from the toolbox
robot = rtb.models.DH.Stanford()
print(robot)
print("\n--- Denavit-Hartenberg Parameters ---")
print(robot.dyntable())

In [ ]:
# Display the DH table in a more readable form
print("Link parameters (DH convention):")
print(f"{'Joint':>6} {'theta':>8} {'d':>8} {'a':>8} {'alpha':>8} {'type':>6}")
print("-" * 50)
for i, link in enumerate(robot.links):
    jtype = 'P' if link.isprismatic else 'R'
    print(f"{i+1:>6} {link.theta:>8.4f} {link.d:>8.4f} {link.a:>8.4f} {link.alpha:>8.4f} {jtype:>6}")

## 2. Specify dynamic parameters

Below we specify/verify the dynamic parameters for each link of the Stanford manipulator:  
masses, centre-of-mass positions, inertia tensors, motor inertias, viscous and Coulomb friction  
coefficients, gear ratios, and joint limits.

The values below are based on the Robotics Toolbox built-in model (Corke, 2017) and supplemented  
with representative values where needed.

In [ ]:
# Display existing dynamic parameters
print("=== Dynamic Parameters of the Stanford Manipulator ===")
print()
for i, link in enumerate(robot.links):
    print(f"--- Link {i+1} ({'Prismatic' if link.isprismatic else 'Revolute'}) ---")
    print(f"  Mass (m):               {link.m:.4f} kg")
    print(f"  Centre of mass (r):     {link.r}")
    print(f"  Inertia tensor (I):     {link.I}")
    print(f"  Motor inertia (Jm):     {link.Jm}")
    print(f"  Viscous friction (B):   {link.B}")
    print(f"  Coulomb friction (Tc):  {link.Tc}")
    print(f"  Gear ratio (G):         {link.G}")
    print(f"  Joint limits (qlim):    {link.qlim}")
    print()

In [ ]:
# If any dynamic parameters are missing/zero, we set realistic values here.
# (The built-in Stanford model already has masses and inertias from Corke's data.)
# We ensure friction, motor inertia, and gear ratios are set.

for i, link in enumerate(robot.links):
    # Set motor inertia if zero
    if link.Jm == 0:
        link.Jm = 200e-6  # 200 µkg·m² (typical small servo motor)
    # Set gear ratio if 1 (i.e., not set)
    if link.G == 1:
        link.G = 100 if not link.isprismatic else 50  # typical harmonic drive ratios
    # Set viscous friction if zero
    if link.B == 0:
        link.B = 0.01  # N·m·s/rad
    # Set Coulomb friction if zero
    if np.all(np.array(link.Tc) == 0):
        link.Tc = [0.4, -0.4]  # [positive direction, negative direction]
    # Set joint limits if not set
    if link.qlim is None or np.all(np.array(link.qlim) == 0):
        if link.isprismatic:
            link.qlim = [0.0, 1.0]  # metres
        else:
            link.qlim = [-np.pi, np.pi]  # radians

print("Dynamic parameters updated (where defaults were missing).")
print()

# Verify by re-printing
for i, link in enumerate(robot.links):
    print(f"Link {i+1}: m={link.m:.3f} kg, Jm={link.Jm:.2e}, G={link.G}, B={link.B:.4f}, Tc={link.Tc}")

## 3. Specify and display initial and final configurations

In [ ]:
# Initial configuration (q_start) and final configuration (q_end)
# For Stanford: joints are [R, R, P, R, R, R]
#   Joint 3 is prismatic (in metres), the rest are revolute (in radians)

q_start = np.array([0.0, 0.3, 0.3, 0.0, 0.0, 0.0])     # near-home position
q_end   = np.array([1.0, -0.5, 0.6, 0.8, -0.3, 0.5])    # arbitrary target

print("Initial configuration q_start =", q_start)
print("Final   configuration q_end   =", q_end)

# Forward kinematics for both configurations
T_start = robot.fkine(q_start)
T_end   = robot.fkine(q_end)

print("\nEnd-effector pose at q_start:")
print(T_start)
print("\nEnd-effector pose at q_end:")
print(T_end)

In [ ]:
# Visualise both configurations
fig, axes = plt.subplots(1, 2, subplot_kw={'projection': '3d'}, figsize=(14, 6))

robot.plot(q_start, backend='pyplot', fig=fig, ax=axes[0], block=False)
axes[0].set_title('Initial configuration (q_start)')

robot.plot(q_end, backend='pyplot', fig=fig, ax=axes[1], block=False)
axes[1].set_title('Final configuration (q_end)')

plt.tight_layout()
plt.show()

## 4. Plan a trajectory between configurations

We use a 5th-order polynomial (quintic) trajectory in joint space.  
This ensures smooth positions, velocities, and accelerations with zero boundary conditions.

In [ ]:
# Trajectory parameters
T_total = 2.0    # total time [s]
dt = 0.01        # time step [s]
t = np.arange(0, T_total + dt, dt)  # time vector
N = len(t)

# Generate joint-space trajectory using quintic polynomial
traj = rtb.jtraj(q_start, q_end, t)

q   = traj.q      # joint positions  (N x n)
qd  = traj.qd     # joint velocities (N x n)
qdd = traj.qdd    # joint accelerations (N x n)

print(f"Trajectory: {N} time steps, {q.shape[1]} joints")
print(f"Time range: {t[0]:.2f} to {t[-1]:.2f} s")

In [ ]:
# Plot joint trajectories: positions, velocities, accelerations
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
joint_labels = [f'Joint {i+1}' for i in range(q.shape[1])]

for j in range(q.shape[1]):
    axes[0].plot(t, q[:, j], label=joint_labels[j])
    axes[1].plot(t, qd[:, j], label=joint_labels[j])
    axes[2].plot(t, qdd[:, j], label=joint_labels[j])

axes[0].set_ylabel('Position [rad / m]')
axes[0].set_title('Joint Positions')
axes[0].legend(loc='upper left', fontsize=8)
axes[0].grid(True)

axes[1].set_ylabel('Velocity [rad/s / m/s]')
axes[1].set_title('Joint Velocities')
axes[1].legend(loc='upper left', fontsize=8)
axes[1].grid(True)

axes[2].set_ylabel('Acceleration [rad/s² / m/s²]')
axes[2].set_title('Joint Accelerations')
axes[2].set_xlabel('Time [s]')
axes[2].legend(loc='upper left', fontsize=8)
axes[2].grid(True)

plt.tight_layout()
plt.show()

## 5. Inverse dynamics using Newton–Euler method

The equation of motion for a rigid-body manipulator is:

$$\tau = M(q)\ddot{q} + C(q, \dot{q})\dot{q} + G(q)$$

We solve the inverse dynamics for three scenarios:
1. **Full dynamics:** $\dot{q} \neq 0,\; \ddot{q} \neq 0$
2. **Quasi-statics:** $\dot{q} \neq 0,\; \ddot{q} \approx 0$
3. **Static (holding position):** $\dot{q} = 0,\; \ddot{q} = 0$

In [ ]:
n_joints = q.shape[1]

# Pre-allocate torque arrays for all 3 scenarios
tau_full   = np.zeros((N, n_joints))   # scenario 1: full dynamics
tau_quasi  = np.zeros((N, n_joints))   # scenario 2: quasi-statics
tau_static = np.zeros((N, n_joints))   # scenario 3: holding position

for k in range(N):
    # Scenario 1: full dynamics  (q̈ ≠ 0, q̇ ≠ 0)
    tau_full[k, :] = robot.rne(q[k, :], qd[k, :], qdd[k, :])
    
    # Scenario 2: quasi-statics  (q̈ ≈ 0, q̇ ≠ 0)
    tau_quasi[k, :] = robot.rne(q[k, :], qd[k, :], np.zeros(n_joints))
    
    # Scenario 3: static hold    (q̈ = 0, q̇ = 0)
    tau_static[k, :] = robot.rne(q[k, :], np.zeros(n_joints), np.zeros(n_joints))

print("Inverse dynamics computed for all 3 scenarios.")
print(f"  tau_full   shape: {tau_full.shape}")
print(f"  tau_quasi  shape: {tau_quasi.shape}")
print(f"  tau_static shape: {tau_static.shape}")

## 6. Compute M(q), C(q, q̇), G(q) matrices at each time step

In [ ]:
# Store matrices at each time step
M_list = []   # inertia matrix M(q)
C_list = []   # Coriolis/centrifugal matrix C(q, q̇)
G_list = []   # gravity vector G(q)

for k in range(N):
    M_q   = robot.inertia(q[k, :])                   # n x n
    C_qqd = robot.coriolis(q[k, :], qd[k, :])        # n x n
    G_q   = robot.gravload(q[k, :])                   # n-vector
    
    M_list.append(M_q)
    C_list.append(C_qqd)
    G_list.append(G_q)

M_arr = np.array(M_list)  # (N, n, n)
C_arr = np.array(C_list)  # (N, n, n)
G_arr = np.array(G_list)  # (N, n)

print(f"M(q) shape:      {M_arr.shape}")
print(f"C(q,q̇) shape:   {C_arr.shape}")
print(f"G(q) shape:      {G_arr.shape}")

In [ ]:
# Print example values at the midpoint of the trajectory
mid = N // 2
print(f"\n=== Values at t = {t[mid]:.2f} s (trajectory midpoint) ===")
print(f"\nq = {q[mid, :]}")
print(f"q̇ = {qd[mid, :]}")
print(f"q̈ = {qdd[mid, :]}")

print(f"\nM(q) = \n{M_arr[mid]}")
print(f"\nC(q, q̇) = \n{C_arr[mid]}")
print(f"\nG(q) = {G_arr[mid]}")

In [ ]:
# Verify: tau = M*qdd + C*qd + G  should match tau_full
tau_verify = np.zeros((N, n_joints))
for k in range(N):
    tau_verify[k, :] = M_arr[k] @ qdd[k, :] + C_arr[k] @ qd[k, :] + G_arr[k]

error = np.max(np.abs(tau_full - tau_verify))
print(f"\nVerification: max |tau_rne - (M*qdd + C*qd + G)| = {error:.2e}")
print("(Should be ≈ 0 if friction terms are excluded from both sides.)")

In [ ]:
# Plot selected elements of M(q) over time to show how inertia changes along the trajectory
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Diagonal elements of M (effective inertias)
for j in range(min(n_joints, 4)):
    ax = axes[j // 2, j % 2]
    ax.plot(t, M_arr[:, j, j], 'b-', linewidth=1.5)
    ax.set_title(f'M[{j+1},{j+1}](q) — Effective inertia of Joint {j+1}')
    ax.set_xlabel('Time [s]')
    ax.set_ylabel('Inertia')
    ax.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Plot gravity vector components over time
fig, ax = plt.subplots(figsize=(12, 5))
for j in range(n_joints):
    ax.plot(t, G_arr[:, j], label=f'G[{j+1}]')
ax.set_title('Gravity load G(q) components along the trajectory')
ax.set_xlabel('Time [s]')
ax.set_ylabel('Torque / Force')
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

## 7. Plot joint torques for each scenario

In [ ]:
# --- Scenario 1: Full dynamics ---
fig, axes = plt.subplots(n_joints, 1, figsize=(12, 2.5 * n_joints), sharex=True)
fig.suptitle('Scenario 1: Full Dynamics ($\\dot{q} \\neq 0$, $\\ddot{q} \\neq 0$)',
             fontsize=14, y=1.01)

for j in range(n_joints):
    unit = 'N' if robot.links[j].isprismatic else 'N·m'
    axes[j].plot(t, tau_full[:, j], 'b-', linewidth=1.2)
    axes[j].set_ylabel(f'Joint {j+1} [{unit}]')
    axes[j].grid(True)

axes[-1].set_xlabel('Time [s]')
plt.tight_layout()
plt.show()

In [ ]:
# --- Scenario 2: Quasi-statics ---
fig, axes = plt.subplots(n_joints, 1, figsize=(12, 2.5 * n_joints), sharex=True)
fig.suptitle('Scenario 2: Quasi-statics ($\\dot{q} \\neq 0$, $\\ddot{q} \\approx 0$)',
             fontsize=14, y=1.01)

for j in range(n_joints):
    unit = 'N' if robot.links[j].isprismatic else 'N·m'
    axes[j].plot(t, tau_quasi[:, j], 'r-', linewidth=1.2)
    axes[j].set_ylabel(f'Joint {j+1} [{unit}]')
    axes[j].grid(True)

axes[-1].set_xlabel('Time [s]')
plt.tight_layout()
plt.show()

In [ ]:
# --- Scenario 3: Static hold ---
fig, axes = plt.subplots(n_joints, 1, figsize=(12, 2.5 * n_joints), sharex=True)
fig.suptitle('Scenario 3: Static hold ($\\dot{q} = 0$, $\\ddot{q} = 0$)',
             fontsize=14, y=1.01)

for j in range(n_joints):
    unit = 'N' if robot.links[j].isprismatic else 'N·m'
    axes[j].plot(t, tau_static[:, j], 'g-', linewidth=1.2)
    axes[j].set_ylabel(f'Joint {j+1} [{unit}]')
    axes[j].grid(True)

axes[-1].set_xlabel('Time [s]')
plt.tight_layout()
plt.show()

In [ ]:
# --- Combined comparison for each joint ---
fig, axes = plt.subplots(n_joints, 1, figsize=(14, 2.5 * n_joints), sharex=True)
fig.suptitle('Comparison of all three scenarios', fontsize=14, y=1.01)

for j in range(n_joints):
    unit = 'N' if robot.links[j].isprismatic else 'N·m'
    axes[j].plot(t, tau_full[:, j],   'b-',  label='Full dynamics', linewidth=1.2)
    axes[j].plot(t, tau_quasi[:, j],  'r--', label='Quasi-statics', linewidth=1.2)
    axes[j].plot(t, tau_static[:, j], 'g:',  label='Static hold',  linewidth=1.5)
    axes[j].set_ylabel(f'Joint {j+1} [{unit}]')
    axes[j].grid(True)
    if j == 0:
        axes[j].legend(loc='upper right', fontsize=8)

axes[-1].set_xlabel('Time [s]')
plt.tight_layout()
plt.show()

## 8. Analysis and conclusions

### Key observations

**1. Full dynamics vs quasi-statics:**  
The difference between the full-dynamics torques (blue) and quasi-static torques (red) represents  
the contribution of the inertial term $M(q)\ddot{q}$. At the start and end of the trajectory,  
where accelerations are highest (due to the quintic polynomial), this difference is most pronounced.  
At mid-trajectory, where $\ddot{q} \approx 0$, the two curves converge.

**2. Quasi-statics vs static hold:**  
The difference between quasi-static (red) and static (green) torques reflects the Coriolis and  
centrifugal terms $C(q, \dot{q})\dot{q}$. These are velocity-dependent and peak at mid-trajectory  
where velocities are maximal.

**3. Static torques:**  
The green curves show the gravity compensation torques $G(q)$ that the robot must apply simply  
to maintain each configuration along the trajectory. These change smoothly as the configuration  
evolves.

**4. Inertia matrix M(q):**  
The diagonal elements of $M(q)$ show how the effective inertia of each joint varies with  
configuration. Joint 1 (base rotation) typically has the largest inertia as it must move the  
entire kinematic chain.

**5. Prismatic joint (Joint 3):**  
The Stanford arm has a prismatic third joint. Its "torque" is actually a force (in Newtons),  
and its gravity component is a constant upward force to support the arm weight.

### Summary of the equation of motion decomposition

| Term | Physical meaning | When it matters |
|------|-----------------|----------------|
| $M(q)\ddot{q}$ | Inertial torques | High accelerations (start/end of motion) |
| $C(q,\dot{q})\dot{q}$ | Coriolis & centrifugal | High velocities (mid-trajectory) |
| $G(q)$ | Gravity compensation | Always (configuration-dependent) |